# Analisis Data Lowongan Digital untuk LokerLens

In [1]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    sns = None


In [2]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "data", "processed", "lokerlens_dataset_labeled.csv")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
df = pd.read_csv(DATA_PATH)
df["text_length"] = df["job_text"].fillna("").astype(str).str.len()
df.shape


(300, 37)

## Data Overview

In [3]:
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(20))
print("Duplicate rows:", df.duplicated().sum())
display(df["source_name"].value_counts())
display(df["source_platform"].value_counts())
display(df["risk_level"].value_counts())


,id,source_name,source_platform,source_url,collected_at,job_title,company_name,company_clarity,job_type,job_text,...,unrealistic_promise_terms,workload_status,missing_fields,red_flag_categories,red_flag_count,trust_score,risk_level,recommended_action,notes,text_length
0,1,arbeitnow,job_site,https://www.arbeitnow.com/jobs/companies/softg...,2026-07-07T15:20:14,Senior Game Producer (Casual Games) - Fully Re...,SOFTGAMES,clear,unknown,Senior Game Producer (Casual Games) - Fully Re...,...,NaN,heavy,duration;contact;contract;deadline,NaN,0,84,"Cukup Aman, Tapi Perlu Dicek",Apply setelah klarifikasi,public_api_arbeitnow; endpoint=https://www.arb...,4033
1,2,arbeitnow,job_site,https://www.arbeitnow.com/jobs/companies/sumup...,2026-07-07T13:45:25,Senior Analytics Engineer - Run & Grow,SumUp,clear,unknown,Senior Analytics Engineer - Run & Grow SumUp T...,...,NaN,unclear,working_hours;work_arrangement;duration;contac...,working_hours_missing,1,72,"Cukup Aman, Tapi Perlu Dicek",Apply setelah klarifikasi,public_api_arbeitnow; endpoint=https://www.arb...,5694
2,3,arbeitnow,job_site,https://www.arbeitnow.com/jobs/companies/onaps...,2026-07-07T13:45:25,SAP ABAP Software Engineer (m/f/d),Onapsis,clear,unknown,SAP ABAP Software Engineer (m/f/d) Onapsis Abo...,...,NaN,unclear,compensation;working_hours;duration;contact;co...,compensation_missing;working_hours_missing,2,67,Berisiko Sedang,Jangan apply dulu sebelum verifikasi,public_api_arbeitnow; endpoint=https://www.arb...,4993
3,4,arbeitnow,job_site,https://www.arbeitnow.com/jobs/companies/curre...,2026-07-07T13:20:58,Senior Account Executive (m/w/d),CURRENTA GRUPPE,clear,unknown,Senior Account Executive (m/w/d) CURRENTA GRUP...,...,NaN,unclear,compensation;working_hours;work_arrangement;du...,compensation_missing;working_hours_missing,2,62,Berisiko Sedang,Jangan apply dulu sebelum verifikasi,public_api_arbeitnow; endpoint=https://www.arb...,2172
4,5,arbeitnow,job_site,https://www.arbeitnow.com/jobs/companies/conne...,2026-07-07T11:20:17,IT-Systemadministrator (m/w/d),connect people & company GmbH,clear,unknown,IT-Systemadministrator (m/w/d) connect people ...,...,NaN,unclear,compensation;working_hours;work_arrangement;du...,compensation_missing;working_hours_missing,2,62,Berisiko Sedang,Jangan apply dulu sebelum verifikasi,public_api_arbeitnow; endpoint=https://www.arb...,3514


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         300 non-null    int64  
 1   source_name                300 non-null    object 
 2   source_platform            300 non-null    object 
 3   source_url                 300 non-null    object 
 4   collected_at               300 non-null    object 
 5   job_title                  300 non-null    object 
 6   company_name               300 non-null    object 
 7   company_clarity            300 non-null    object 
 8   job_type                   300 non-null    object 
 9   job_text                   300 non-null    object 
 10  job_description            300 non-null    object 
 11  requirements               91 non-null     object 
 12  benefits                   130 non-null    object 
 13  compensation_status        300 non-null    object 

None

unrealistic_promise_terms    300
sensitive_data_terms         300
payment_terms                299
duration_text                231
working_hours_text           213
requirements                 209
benefits                     170
compensation_text            152
red_flag_categories           30
location                       8
id                             0
source_platform                0
source_name                    0
collected_at                   0
source_url                     0
company_clarity                0
job_type                       0
job_text                       0
job_description                0
job_title                      0
dtype: int64

Duplicate rows: 0


source_name
arbeitnow    68
themuse      42
remoteok     42
linkedin     42
jobstreet    42
dealls       36
remotive     28
Name: count, dtype: int64

source_platform
job_site    258
linkedin     42
Name: count, dtype: int64

risk_level
Berisiko Sedang                 193
Cukup Aman, Tapi Perlu Dicek    105
Risiko Tinggi                     1
Aman Dilamar                      1
Name: count, dtype: int64

## EDA dan Visualisasi

In [4]:
def split_terms(series):
    counter = Counter()
    for value in series.fillna(""):
        for item in str(value).split(";"):
            item = item.strip()
            if item:
                counter[item] += 1
    return counter

def save_bar(series, filename, title, xlabel, ylabel, rotate=False, color="#173b57"):
    plt.figure(figsize=(8, 5))
    series.fillna("not_mentioned").value_counts().plot(kind="bar", color=color)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=35 if rotate else 0, ha="right" if rotate else "center")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300)
    plt.close()


In [5]:
save_bar(df["source_platform"], "01_source_platform_distribution.png", "Distribusi Platform Sumber Lowongan", "Platform", "Jumlah Lowongan", True)
save_bar(df["job_type"], "02_job_type_distribution.png", "Distribusi Jenis Lowongan", "Jenis Lowongan", "Jumlah Lowongan", True)
pd.Series(split_terms(df["missing_fields"])).sort_values(ascending=False).plot(kind="bar", figsize=(8, 5), color="#b45309", title="Frekuensi Field yang Tidak Disebutkan")
plt.xlabel("Missing Field")
plt.ylabel("Frekuensi")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "03_missing_information_frequency.png"), dpi=300)
plt.close()


In [6]:
pd.Series(split_terms(df["red_flag_categories"])).sort_values(ascending=False).plot(kind="bar", figsize=(8, 5), color="#dc2626", title="Frekuensi Kategori Red Flag")
plt.xlabel("Red Flag")
plt.ylabel("Frekuensi")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "04_red_flag_frequency.png"), dpi=300)
plt.close()
save_bar(df["risk_level"], "05_risk_level_distribution.png", "Distribusi Risk Level", "Risk Level", "Jumlah Lowongan", True, "#475569")
save_bar(df["compensation_status"], "06_compensation_status.png", "Distribusi Status Kompensasi", "Status Kompensasi", "Jumlah Lowongan", True, "#0f766e")


In [7]:
pd.crosstab(df["source_name"], df["risk_level"]).plot(kind="bar", stacked=True, figsize=(8, 5), colormap="viridis")
plt.title("Sumber Data vs Risk Level")
plt.xlabel("Sumber Data")
plt.ylabel("Jumlah Lowongan")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "07_platform_vs_risk_level.png"), dpi=300)
plt.close()


In [8]:
risk_order = ["Aman Dilamar", "Cukup Aman, Tapi Perlu Dicek", "Berisiko Sedang", "Risiko Tinggi"]
values = [df.loc[df["risk_level"] == level, "text_length"].values for level in risk_order if (df["risk_level"] == level).any()]
labels = [level for level in risk_order if (df["risk_level"] == level).any()]
plt.figure(figsize=(8, 5))
plt.boxplot(values, tick_labels=labels)
plt.title("Panjang Teks Lowongan Berdasarkan Risk Level")
plt.xlabel("Risk Level")
plt.ylabel("Panjang Teks")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "08_text_length_by_risk_level.png"), dpi=300)
plt.close()


## Text Analysis

In [9]:
stopwords = set("and the for with from this that you your are our will have has job work role team company serta yang untuk dengan dari pada dalam adalah lowongan kerja".split())
def top_words(frame, limit=12):
    words = []
    for text in frame["job_text"].fillna("").astype(str):
        words.extend([word for word in re.findall(r"[a-zA-Z]{4,}", text.lower()) if word not in stopwords])
    return Counter(words).most_common(limit)
risky = df[df["risk_level"].isin(["Berisiko Sedang", "Risiko Tinggi"])]
safe = df[df["risk_level"] == "Aman Dilamar"]
top_risky_words = top_words(risky if not risky.empty else df)
top_safe_words = top_words(safe if not safe.empty else df)
display(pd.DataFrame(top_risky_words, columns=["keyword", "risky_frequency"]))
display(pd.DataFrame(top_safe_words, columns=["keyword", "safe_frequency"]))


,keyword,risky_frequency
0,lalu,271
1,aktif,266
2,site,244
3,rekruter,232
4,intern,201
5,experience,200
6,jakarta,194
7,freelance,189
8,support,157
9,negotiable,152


,keyword,safe_frequency
0,experience,10
1,time,6
2,full,6
3,benefits,6
4,generalist,4
5,years,4
6,required,4
7,desired,4
8,plus,4
9,payroll,3


In [10]:
pd.Series(dict(top_risky_words)).sort_values().plot(kind="barh", figsize=(8, 5), color="#ea580c", title="Top Keyword pada Lowongan Berisiko")
plt.xlabel("Frekuensi")
plt.ylabel("Keyword")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "09_top_keywords_red_flag.png"), dpi=300)
plt.close()


In [11]:
feature_df = pd.DataFrame({
    "trust_score": df["trust_score"],
    "red_flag_count": df["red_flag_count"],
    "text_length": df["text_length"],
    "asks_payment": (df["asks_payment"] == "yes").astype(int),
    "asks_sensitive_data": (df["asks_sensitive_data"] == "yes").astype(int),
    "unrealistic_promise": (df["unrealistic_promise"] == "yes").astype(int),
    "missing_count": df["missing_fields"].fillna("").map(lambda x: len([item for item in str(x).split(";") if item])),
})
corr = feature_df.corr(numeric_only=True).fillna(0)
plt.figure(figsize=(8, 5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=35, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Heatmap Korelasi Fitur Numerik dan Biner")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "10_correlation_heatmap.png"), dpi=300)
plt.close()


## Insight Utama

- Kompensasi tidak disebutkan muncul pada 211 lowongan, sehingga transparansi imbalan menjadi isu utama untuk proses apply.
- Jam kerja tidak jelas atau tidak disebutkan muncul pada 251 lowongan, yang membuat estimasi beban kerja perlu diklarifikasi.
- Sumber data terbesar berasal dari arbeitnow dengan 200 lowongan, sehingga interpretasi platform harus mempertimbangkan dominasi sumber tersebut.
- Red flag paling sering adalah working_hours_missing dengan 251 kemunculan.
- Risk level dengan Trust Score rata-rata terendah adalah Risiko Tinggi (32.0), menunjukkan hubungan missing information dan red flag terhadap skor.


## Critical Analysis

Mahasiswa, fresh graduate, dan freelancer pemula rentan terhadap lowongan tidak transparan karena sering membutuhkan pengalaman awal dan belum selalu punya pembanding proses rekrutmen yang sehat. Informasi kompensasi, jam kerja, kontrak, kontak resmi, dan supervisor penting karena menentukan beban kerja, batas komitmen, dan keamanan data pelamar. Permintaan data pribadi atau pembayaran terlalu awal perlu diperlakukan sebagai indikasi risiko yang harus diverifikasi.

## Implikasi untuk LokerLens

Hasil EDA mendukung Trust Score, Missing Information Checker, Red Flag Detector, Safe Apply Checklist, dan Recruiter Question Generator. Pola missing information dapat langsung diubah menjadi pertanyaan klarifikasi untuk recruiter dan bahan edukasi apply yang lebih aman.

In [12]:
summary_path = os.path.join(REPORTS_DIR, "insight_summary.md")
print(f"Insight summary tersimpan di {summary_path}")


Insight summary tersimpan di c:\Users\firma\Documents\Python\Kuliah\AKB\LokerLens\lokerlens-data\reports\insight_summary.md
